## Evaluation (Baseline Paper)

This notebook evaluates all trained models using:
- Macro-F1 over 18 gesture classes
- Binary F1 (BFRB vs non-BFRB)
- Macro-F1 over 9 classes (8 BFRB gestures + one merged non-BFRB)

All model predictions/logits are loaded from saved `.pt` artifacts produced in `04_models_paper.ipynb`.


## 1. Environment setup 

We import all necessary libraries for the evaluation.

In [4]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import f1_score, confusion_matrix, classification_report

## Load
We loaded all the processed files and sequences for the evaluation.

In [5]:
DATA_PROCESSED = Path("../data/processed")

with open(DATA_PROCESSED / "train_clean.pkl", "rb") as f:
    train_df = pickle.load(f)

with open(DATA_PROCESSED / "test_clean.pkl", "rb") as f:
    test_df = pickle.load(f)

print("Train sequences:", train_df["sequence_id"].nunique())
print("Test sequences :", test_df["sequence_id"].nunique())
print(train_df[["gesture", "behavior"]].head())

Train sequences: 6515
Test sequences : 1629
              gesture                                   behavior
0  Cheek - pinch skin  Relaxes and moves hand to target location
1  Cheek - pinch skin  Relaxes and moves hand to target location
2  Cheek - pinch skin  Relaxes and moves hand to target location
3  Cheek - pinch skin  Relaxes and moves hand to target location
4  Cheek - pinch skin  Relaxes and moves hand to target location


## BFRB Gesture Definition
We define the set of gestures considered as BFRB-related. All other gestures are treated as non-BFRB for binary detection and merged into a single class for macro-(K+1) evaluation.

In [6]:
BFRB_GESTURES = {
    'Cheek - pinch skin',
    'Forehead - pull hairline',
    'Neck - scratch',
    'Neck - pinch skin',
    'Eyelash - pull hair',
    'Eyebrow - pull hair',
    'Forehead - scratch',
    'Above ear - pull hair',
    'Pinch knee/leg skin',
    'Scratch knee/leg skin',
}

print("BFRB gesture count:", len(BFRB_GESTURES))

BFRB gesture count: 10


In [7]:
def load_outputs(filename: str):
    path = DATA_PROCESSED / filename
    return torch.load(path, weights_only=False)

## Evaluation Metrics


We compute:

- Accuracy
- Macro-F1 over 18 gesture classes
- Binary F1 for BFRB detection
- Macro-F1 over K+1 classes (BFRB gestures + merged non-BFRB)

These metrics allow both fine-grained and coarse-grained evaluation.



In [8]:
def compute_metrics(outputs, bfrb_gestures):
    y_true = outputs["y_true"].cpu().numpy()
    y_pred = outputs["y_pred"].cpu().numpy()
    classes = [str(c) for c in outputs["classes"]]

    # 18-class
    acc = float((y_pred == y_true).mean())
    macro18 = float(f1_score(y_true, y_pred, average="macro"))

    # ids -> gesture names
    y_true_g = np.array([classes[i] for i in y_true], dtype=object)
    y_pred_g = np.array([classes[i] for i in y_pred], dtype=object)

    # Binary BFRB detection
    y_true_bin = np.array([1 if g in bfrb_gestures else 0 for g in y_true_g])
    y_pred_bin = np.array([1 if g in bfrb_gestures else 0 for g in y_pred_g])

    # Handle degenerate case safely
    binary_f1 = float("nan") if len(np.unique(y_true_bin)) < 2 else float(f1_score(y_true_bin, y_pred_bin))

    # Macro-(K+1): each BFRB gesture as its own class + merged non_BFRB
    def to_kplus1(g):
        return g if g in bfrb_gestures else "non_BFRB"
    
    y_true_k = np.array([to_kplus1(g) for g in y_true_g], dtype=object)
    y_pred_k = np.array([to_kplus1(g) for g in y_pred_g], dtype=object)

    macro_kplus1 = float("nan") if len(np.unique(y_true_k)) < 2 else float(f1_score(y_true_k, y_pred_k, average="macro"))

    return {
        "acc": acc,
        "macro18": macro18,
        "binary_f1": binary_f1,
        "macro_kplus1": macro_kplus1,
        "frac_bfrb_true": float(y_true_bin.mean()),
        "frac_bfrb_pred": float(y_pred_bin.mean()),
    }

## Model Comparison

The table below summarizes performance across all six models.


In [15]:
model_files = {
    "M1 FFT-MLP all": "model1_fft_mlp_all_outputs.pt",
    "M2 FFT-MLP IMU+THM": "model2_fft_mlp_imu_thm_outputs.pt",
    "M3 CNN-BiLSTM TOF": "model3_cnn_bilstm_tof_outputs.pt",
    "M4 Late Fusion": "model4_late_fusion_outputs.pt",
    "M5 Intermediate Fusion": "model5_intermediate_fusion_outputs.pt",
    "M6 FFT Random Forest": "model6_fft_random_forest_outputs.pt",
}

rows = []
for name, fn in model_files.items():
    out = torch.load(DATA_PROCESSED / fn, weights_only=False)
    metrics = compute_metrics(out, BFRB_GESTURES)
    rows.append({"model": name, **metrics})

results_df = pd.DataFrame(rows)
results_df




,model,acc,macro18,binary_f1,macro_kplus1,frac_bfrb_true,frac_bfrb_pred
0,M1 FFT-MLP all,0.436464,0.445213,0.912612,0.392677,0.658686,0.640884
1,M2 FFT-MLP IMU+THM,0.381829,0.409239,0.906206,0.318367,0.658686,0.656845
2,M3 CNN-BiLSTM TOF,0.360958,0.310532,0.913327,0.314410,0.658686,0.658686
3,M4 Late Fusion,0.487416,0.509534,0.939479,0.418283,0.658686,0.659914
4,M5 Intermediate Fusion,0.429711,0.442005,0.913268,0.369014,0.658686,0.650706
5,M6 FFT Random Forest,0.429711,0.380891,0.902517,0.345682,0.658686,0.682627


In [17]:
results_df.to_csv(DATA_PROCESSED / "model_comparison_results.csv", index=False)
print("Evaluation results saved to data/processed/model_comparison_results.csv")


Evaluation results saved to data/processed/model_comparison_results.csv


## Final Observations

- Late Fusion achieves the highest overall performance.
- Multi-sensor fusion improves both gesture classification and BFRB detection.
- Random Forest provides a competitive classical baseline.
- Binary BFRB detection performance is consistently high across models.

These results align with the findings reported in the baseline paper. 


In [10]:
m4 = load_outputs("model4_late_fusion_outputs.pt")
m4_metrics = compute_metrics(m4, BFRB_GESTURES)

print("\n" + "="*60)
print("Model 4: Late Fusion Ensemble - FINAL RESULTS")
print("="*60)
print(f"Binary F1 (BFRB detection):  {m4_metrics['binary_f1']:.4f}")
print(f"Macro F1 (18 gestures):      {m4_metrics['macro18']:.4f}")
print(f"Accuracy (18 gestures):      {m4_metrics['acc']:.4f}")
print(f"Macro F1 (K+1):              {m4_metrics['macro_kplus1']:.4f}  (K={len(BFRB_GESTURES)})")
print("="*60)



Model 4: Late Fusion Ensemble - FINAL RESULTS
Binary F1 (BFRB detection):  0.9395
Macro F1 (18 gestures):      0.5095
Accuracy (18 gestures):      0.4874
Macro F1 (K+1):              0.4183  (K=10)
